---
title: "Test"
draft: true
---

In [22]:
import argparse
import os
import pathlib

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import scipy.signal
from IPython.display import HTML

from orbit.core.bunch import Bunch
from orbit.core.spacecharge import SpaceChargeCalc2p5D
from orbit.space_charge.sc2p5d import setSC2p5DAccNodes
from orbit.teapot import ContinuousLinearFocusingTEAPOT
from orbit.teapot import TEAPOT_Lattice
from orbit.utils.consts import mass_proton

# local
import pcm
from diag import BunchHistCalc
from utils import intensity_from_perveance
from utils import samp_dist

In [2]:
plt.rcParams["axes.linewidth"] = 1.25
plt.rcParams["xtick.minor.visible"] = True
plt.rcParams["ytick.minor.visible"] = True

In [3]:
args = argparse.Namespace(
    sigma0=80.0,
    eta=0.5,
    r=0.6095,
    eps=1e-6,
    dist="kv",
    length=10.0,
    kin_energy=0.0025,
)

In [4]:
sigma0 = np.radians(args.sigma0)
sigma = sigma0 * args.eta

k0 = sigma0 * 1.0
k = k0 * args.eta

emittance = args.eps * 1.0e-06  # [mm mrad] (4 rms)

eq_radius = pcm.get_eq_radius(emittance, k)
perveance = pcm.get_eq_perveance(emittance, k, k0)

radius = eq_radius * args.r

cov_matrix = np.zeros((6, 6))
cov_matrix[0:4, 0:4] = pcm.get_eq_cov_matrix(radius, emittance)
cov_matrix[4, 4] = args.length**2 / 12.0
cov_matrix_init = cov_matrix.copy()

intensity = intensity_from_perveance(perveance, args.kin_energy, mass_proton, args.length)

In [5]:
# Small correction to envelope oscillation frequency (breathing mode)
env_wave_pred = (2.0 * np.pi) / np.sqrt(2.0 * (1.0 + args.eta**2))  # scaled

periods = 20
history = pcm.track(
    envelope=np.array([args.r, 0.0]),
    particles=np.array([[2.8, 0.0]]),
    eta=args.eta,
    t_max=(periods * env_wave_pred),
    t_steps=(periods * 100),
)

idx, _ = scipy.signal.find_peaks(history["r"])
env_wave_avg = np.mean(np.diff(history["t"][idx]))
env_wave_std = np.std(np.diff(history["t"][idx]))

print("wavelength * k0 (pred) = {:0.4f}".format(env_wave_pred))
print("wavelength * k0 (calc) = {:0.4f} +- {:0.4f}".format(env_wave_avg, env_wave_std))

env_wave = env_wave_avg / k0

# Track test particles for later comparison
history_pcm = pcm.track_strobe(
    envelope=np.array([args.r, 0.0]),
    particles=np.array([[2.8, 0.0]]),  # ~separatrix
    eta=args.eta,
    periods=500,
)

wavelength * k0 (pred) = 3.9738
wavelength * k0 (calc) = 3.8566 +- 0.0087


In [6]:
particles = np.zeros((100_000, 6))
particles[:, :4] = samp_dist(size=particles.shape[0], name=args.dist, cov_matrix=cov_matrix_init[:4, :4])
particles[:, 4] = args.length * np.random.uniform(-0.5, 0.5, size=particles.shape[0])

bunch = Bunch()
bunch.mass(mass_proton)
bunch.getSyncParticle().kinEnergy(args.kin_energy)
bunch.macroSize(intensity / particles.shape[0])
for i in range(particles.shape[0]):
    bunch.addParticle(*particles[i])

lattice = TEAPOT_Lattice()
lattice.addNode(
    ContinuousLinearFocusingTEAPOT(length=env_wave, kq=k0**2, nparts=30)
)

sc_calc = SpaceChargeCalc2p5D(128, 128, 1)
sc_path_length_min = 0.001
sc_nodes = setSC2p5DAccNodes(lattice, sc_path_length_min, sc_calc)

In [7]:
scale = np.array([eq_radius, eq_radius * k0])
xmax = 3.5 * scale
limits = list(zip(-xmax, xmax))

hist_calc = BunchHistCalc(
    axis=(0, 1),
    shape=(150, 150),
    limits=limits,
)

histograms = []
for period in range(26):
    if period > 0:
        lattice.trackBunch(bunch)

    histograms.append(hist_calc(bunch))

In [21]:
import matplotlib.animation as animation
from IPython.display import HTML

histogram = histograms[0].copy()


fig, ax = plt.subplots(figsize=(5, 4))
mesh = ax.pcolormesh(
    histogram.edges[0] / scale[0],
    histogram.edges[1] / scale[1],
    np.ma.log10(histogram.values.T / np.max(histogram.values)),
    cmap="Greys",
    vmin=-3,
)
ax.scatter(
    history_pcm["particles"][..., 0],
    history_pcm["particles"][..., 1],
    ec="none",
    s=1,
    c="red",
)
ax.set_xlabel("$x / r_0$")
ax.set_ylabel(r"$x' / k_0 r_0$")
fig.colorbar(mesh, ax=ax)
plt.close()

def update(frame):
    histogram = histograms[frame].copy()
    values = histogram.values
    values = values / np.max(values)
    values = np.ma.masked_less_equal(values, 0.0, None)
    values = np.ma.log10(values)
    
    mesh.set_array(values.ravel(order="F"))    

    ax.set_title(f"Period = {frame:02.0f}", fontsize="medium")

    return [mesh]

anim = animation.FuncAnimation(
    fig, 
    update, 
    frames=len(histograms), 
    interval=100, 
    blit=True
)
HTML(anim.to_jshtml())